In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install torch torchvision torchaudio -q
!pip install torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 796.5 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 7.0 MB/s eta 0:00:00a 0:00:01


In [3]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from tqdm import tqdm

from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import GCNConv, global_mean_pool


# ============================================================
# CONFIG
# ============================================================

ROOT_DIR = "/kaggle/input/datasets/mdhasanahmad/fresh-water-dataset/Original Image"

BATCH_SIZE_IMG = 32
BATCH_SIZE_GRAPH = 32

EPOCHS = 10
LR = 1e-3

HIDDEN_DIM = 128
DROPOUT = 0.3

SEED = 42

NUM_BOOT = 2000
ALPHA = 0.05


# ============================================================
# DEVICE AND REPRODUCIBILITY
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)


# ============================================================
# 1) DATASET
# ============================================================

class MushroomDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.transform = transform

        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])

        self.images = []
        self.labels = []

        for idx, cls in enumerate(self.classes):

            img_patterns = [
                "*.jpg",
                "*.jpeg",
                "*.png",
                "*.JPG",
                "*.JPEG",
                "*.PNG"
            ]

            for pattern in img_patterns:

                for img_path in glob.glob(
                    os.path.join(root_dir, cls, pattern)
                ):
                    self.images.append(img_path)
                    self.labels.append(idx)

    def __len__(self):

        return len(self.images)

    def __getitem__(self, idx):

        img = Image.open(self.images[idx]).convert("RGB")

        y = self.labels[idx]

        if self.transform:
            img = self.transform(img)

        return img, y


# ============================================================
# CONVNEXT PREPROCESSING
# ============================================================

weights = ConvNeXt_Tiny_Weights.DEFAULT

transform = weights.transforms()


# ============================================================
# LOAD FULL DATASET
# ============================================================

full_dataset = MushroomDataset(
    ROOT_DIR,
    transform=transform
)

num_classes = len(full_dataset.classes)

print("Classes:", full_dataset.classes)
print("Total images:", len(full_dataset))


# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

indices = np.arange(len(full_dataset))

labels = np.array(full_dataset.labels)


# 80% train+validation and 20% test
train_idx, test_idx = train_test_split(

    indices,

    test_size=0.20,

    random_state=SEED,

    stratify=labels
)


# 80% of remaining data for train
# 20% of remaining data for validation

train_idx, val_idx = train_test_split(

    train_idx,

    test_size=0.20,

    random_state=SEED,

    stratify=labels[train_idx]
)


# ============================================================
# SUBSET DATASET
# ============================================================

class SubsetDataset(Dataset):

    def __init__(self, base_ds, idxs):

        self.base_ds = base_ds

        self.idxs = np.array(idxs)

    def __len__(self):

        return len(self.idxs)

    def __getitem__(self, i):

        return self.base_ds[
            int(self.idxs[i])
        ]


train_ds = SubsetDataset(full_dataset, train_idx)

val_ds = SubsetDataset(full_dataset, val_idx)

test_ds = SubsetDataset(full_dataset, test_idx)


# ============================================================
# DATA LOADERS
# ============================================================

train_img_loader = torch.utils.data.DataLoader(

    train_ds,

    batch_size=BATCH_SIZE_IMG,

    shuffle=True,

    num_workers=2,

    pin_memory=True
)


# Separate loader for final training evaluation
train_eval_loader = torch.utils.data.DataLoader(

    train_ds,

    batch_size=BATCH_SIZE_IMG,

    shuffle=False,

    num_workers=2,

    pin_memory=True
)


val_img_loader = torch.utils.data.DataLoader(

    val_ds,

    batch_size=BATCH_SIZE_IMG,

    shuffle=False,

    num_workers=2,

    pin_memory=True
)


test_img_loader = torch.utils.data.DataLoader(

    test_ds,

    batch_size=BATCH_SIZE_IMG,

    shuffle=False,

    num_workers=2,

    pin_memory=True
)


print(
    f"Split: "
    f"train={len(train_ds)} "
    f"val={len(val_ds)} "
    f"test={len(test_ds)}"
)


# ============================================================
# 2) CONVNEXT FEATURE EXTRACTOR
# ============================================================

class ConvNeXtFeatureExtractor(nn.Module):

    def __init__(self):

        super().__init__()

        base = convnext_tiny(
            weights=weights
        )

        self.features = base.features

    def forward(self, x):

        return self.features(x)


convnext_model = ConvNeXtFeatureExtractor().to(device)

convnext_model.eval()


# ============================================================
# FIND FEATURE DIMENSIONS
# ============================================================

with torch.no_grad():

    dummy = torch.randn(
        2,
        3,
        224,
        224
    ).to(device)

    feat = convnext_model(dummy)

    C = feat.shape[1]

    Hf = feat.shape[2]

    Wf = feat.shape[3]


print(
    "ConvNeXt feature map:",
    (C, Hf, Wf)
)


# ============================================================
# 3) BUILD GRID GRAPH
# ============================================================

def build_grid_edge_index(
    H,
    W,
    device="cpu"
):

    edges = []

    for i in range(H):

        for j in range(W):

            idx = i * W + j

            if i + 1 < H:

                edges.append(
                    (idx, idx + W)
                )

                edges.append(
                    (idx + W, idx)
                )

            if j + 1 < W:

                edges.append(
                    (idx, idx + 1)
                )

                edges.append(
                    (idx + 1, idx)
                )

    edge_index = torch.tensor(
        edges,
        dtype=torch.long
    ).t().contiguous().to(device)

    return edge_index


EDGE_INDEX_CPU = build_grid_edge_index(

    Hf,

    Wf,

    device="cpu"
)


# ============================================================
# FEATURE MAP TO GRAPH
# ============================================================

def features_to_graph_batch(
    features_cpu,
    labels_cpu
):

    B, C, H, W = features_cpu.shape

    graphs = []

    for b in range(B):

        fm = features_cpu[b]

        x = fm.permute(
            1,
            2,
            0
        ).reshape(
            -1,
            C
        )

        y = torch.tensor(
            [int(labels_cpu[b])],
            dtype=torch.long
        )

        graphs.append(

            Data(

                x=x,

                edge_index=EDGE_INDEX_CPU,

                y=y
            )
        )

    return graphs


# ============================================================
# 4) GCN CLASSIFIER
# ============================================================

class GCNClassifier(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        dropout=0.3
    ):

        super().__init__()

        self.conv1 = GCNConv(
            input_dim,
            hidden_dim
        )

        self.conv2 = GCNConv(
            hidden_dim,
            hidden_dim
        )

        self.lin = nn.Linear(
            hidden_dim,
            output_dim
        )

        self.dropout = dropout

    def forward(self, data):

        x = data.x

        edge_index = data.edge_index

        batch = data.batch

        x = F.relu(
            self.conv1(
                x,
                edge_index
            )
        )

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = F.relu(
            self.conv2(
                x,
                edge_index
            )
        )

        x = global_mean_pool(
            x,
            batch
        )

        logits = self.lin(x)

        return logits


gcn_model = GCNClassifier(

    input_dim=C,

    hidden_dim=HIDDEN_DIM,

    output_dim=num_classes,

    dropout=DROPOUT

).to(device)


optimizer = torch.optim.AdamW(

    gcn_model.parameters(),

    lr=LR
)


criterion = nn.CrossEntropyLoss()


# ============================================================
# 5) PREDICTION FUNCTION
# ============================================================

@torch.no_grad()
def predict_on_loader(img_loader):

    gcn_model.eval()

    convnext_model.eval()

    all_y = []

    all_pred = []

    all_prob = []

    for imgs, ys in tqdm(

        img_loader,

        desc="Predict"

    ):

        imgs = imgs.to(

            device,

            non_blocking=True
        )

        ys = ys.to(

            device,

            non_blocking=True
        )

        feats = convnext_model(imgs)

        graphs = features_to_graph_batch(

            feats.cpu(),

            ys.cpu()
        )

        batch = Batch.from_data_list(

            graphs

        ).to(device)

        logits = gcn_model(batch)

        probs = F.softmax(

            logits,

            dim=1
        )

        preds = probs.argmax(

            dim=1
        )

        all_y.append(

            ys.cpu().numpy()
        )

        all_pred.append(

            preds.cpu().numpy()
        )

        all_prob.append(

            probs.cpu().numpy()
        )


    y_true = np.concatenate(all_y)

    y_pred = np.concatenate(all_pred)

    y_prob = np.concatenate(all_prob)

    return (

        y_true,

        y_pred,

        y_prob
    )


# ============================================================
# 6) BOOTSTRAP ACCURACY WITH 95% CI
# ============================================================

def bootstrap_accuracy_ci(

    y_true,

    y_pred,

    n_boot=2000,

    alpha=0.05,

    seed=42
):

    rng = np.random.default_rng(seed)

    n = len(y_true)

    point_accuracy = accuracy_score(

        y_true,

        y_pred
    )


    bootstrap_scores = np.zeros(

        n_boot,

        dtype=np.float64
    )


    for i in range(n_boot):

        indices = rng.integers(

            0,

            n,

            size=n
        )

        yt = y_true[indices]

        yp = y_pred[indices]

        bootstrap_scores[i] = accuracy_score(

            yt,

            yp
        )


    lower = np.percentile(

        bootstrap_scores,

        100 * alpha / 2
    )


    upper = np.percentile(

        bootstrap_scores,

        100 * (1 - alpha / 2)
    )


    return (

        point_accuracy,

        lower,

        upper
    )


# ============================================================
# 7) PER-CLASS METRICS WITH 95% CI
# ============================================================

def compute_per_class_metrics_with_ci(

    y_true,

    y_pred,

    y_prob,

    class_names,

    n_boot=2000,

    alpha=0.05,

    seed=42
):

    num_classes = len(class_names)

    n = len(y_true)

    rng = np.random.default_rng(seed)


    # --------------------------------------------------------
    # POINT ESTIMATES
    # --------------------------------------------------------

    prec_classes = precision_score(

        y_true,

        y_pred,

        average=None,

        labels=np.arange(num_classes),

        zero_division=0
    )


    rec_classes = recall_score(

        y_true,

        y_pred,

        average=None,

        labels=np.arange(num_classes),

        zero_division=0
    )


    f1_classes = f1_score(

        y_true,

        y_pred,

        average=None,

        labels=np.arange(num_classes),

        zero_division=0
    )


    auc_classes = []


    for c in range(num_classes):

        yt_c = (

            y_true == c

        ).astype(int)


        if len(np.unique(yt_c)) < 2:

            auc_classes.append(
                float("nan")
            )

        else:

            auc_classes.append(

                roc_auc_score(

                    yt_c,

                    y_prob[:, c]
                )
            )


    auc_classes = np.array(

        auc_classes
    )


    support = np.bincount(

        y_true,

        minlength=num_classes
    )


    # --------------------------------------------------------
    # BOOTSTRAP STORAGE
    # --------------------------------------------------------

    boot_prec = np.zeros(

        (
            n_boot,

            num_classes
        )
    )


    boot_rec = np.zeros(

        (
            n_boot,

            num_classes
        )
    )


    boot_f1 = np.zeros(

        (
            n_boot,

            num_classes
        )
    )


    boot_auc = np.zeros(

        (
            n_boot,

            num_classes
        )
    )


    # --------------------------------------------------------
    # BOOTSTRAP LOOP
    # --------------------------------------------------------

    for i in range(n_boot):

        idx = rng.integers(

            0,

            n,

            size=n
        )


        yt = y_true[idx]

        yp = y_pred[idx]

        ypb = y_prob[idx]


        boot_prec[i] = precision_score(

            yt,

            yp,

            average=None,

            labels=np.arange(num_classes),

            zero_division=0
        )


        boot_rec[i] = recall_score(

            yt,

            yp,

            average=None,

            labels=np.arange(num_classes),

            zero_division=0
        )


        boot_f1[i] = f1_score(

            yt,

            yp,

            average=None,

            labels=np.arange(num_classes),

            zero_division=0
        )


        for c in range(num_classes):

            yt_c = (

                yt == c

            ).astype(int)


            if len(np.unique(yt_c)) < 2:

                boot_auc[i, c] = float(
                    "nan"
                )

            else:

                boot_auc[i, c] = roc_auc_score(

                    yt_c,

                    ypb[:, c]
                )


    # --------------------------------------------------------
    # GET CI
    # --------------------------------------------------------

    def get_ci(arr):

        arr = arr[~np.isnan(arr)]


        if len(arr) == 0:

            return (

                0.0,

                0.0,

                0.0
            )


        mean = float(

            np.mean(arr)
        )


        lo = float(

            np.percentile(

                arr,

                100 * alpha / 2
            )
        )


        hi = float(

            np.percentile(

                arr,

                100 * (1 - alpha / 2)
            )
        )


        return (

            mean,

            lo,

            hi
        )


    # --------------------------------------------------------
    # RESULTS
    # --------------------------------------------------------

    results = []


    for c_idx, cls_name in enumerate(
        class_names
    ):

        p_mean, p_lo, p_hi = get_ci(

            boot_prec[:, c_idx]
        )


        r_mean, r_lo, r_hi = get_ci(

            boot_rec[:, c_idx]
        )


        f_mean, f_lo, f_hi = get_ci(

            boot_f1[:, c_idx]
        )


        a_mean, a_lo, a_hi = get_ci(

            boot_auc[:, c_idx]
        )


        results.append({

            "Class": cls_name,

            "Precision": (

                p_mean,

                p_lo,

                p_hi
            ),

            "Recall": (

                r_mean,

                r_lo,

                r_hi
            ),

            "F1": (

                f_mean,

                f_lo,

                f_hi
            ),

            "AUC": (

                a_mean,

                a_lo,

                a_hi
            ),

            "Support": int(

                support[c_idx]
            )
        })


    return results


# ============================================================
# 8) TRAINING
# ============================================================

for epoch in range(

    1,

    EPOCHS + 1
):

    gcn_model.train()

    total_loss = 0.0

    total_graphs = 0


    for imgs, ys in tqdm(

        train_img_loader,

        desc=f"Epoch {epoch}/{EPOCHS}"
    ):

        imgs = imgs.to(

            device,

            non_blocking=True
        )


        ys = ys.to(

            device,

            non_blocking=True
        )


        with torch.no_grad():

            feats = convnext_model(
                imgs
            )


        graphs = features_to_graph_batch(

            feats.cpu(),

            ys.cpu()
        )


        graph_loader = GeoDataLoader(

            graphs,

            batch_size=BATCH_SIZE_GRAPH,

            shuffle=True
        )


        for batch in graph_loader:

            batch = batch.to(
                device
            )


            optimizer.zero_grad()


            logits = gcn_model(
                batch
            )


            loss = criterion(

                logits,

                batch.y
            )


            loss.backward()


            optimizer.step()


            total_loss += (

                loss.item()

                * batch.num_graphs
            )


            total_graphs += (
                batch.num_graphs
            )


    avg_loss = (

        total_loss

        / max(
            total_graphs,
            1
        )
    )


    print(

        f"Epoch {epoch}: "

        f"TrainLoss={avg_loss:.4f}"
    )


    # --------------------------------------------------------
    # TRAIN ACCURACY
    # --------------------------------------------------------

    y_train, p_train, prob_train = predict_on_loader(

        train_eval_loader
    )


    train_acc = accuracy_score(

        y_train,

        p_train
    )


    print(

        f"Epoch {epoch}: "

        f"TrainAcc={train_acc * 100:.2f}%"
    )


    # --------------------------------------------------------
    # VALIDATION ACCURACY
    # --------------------------------------------------------

    y_val, p_val, prob_val = predict_on_loader(

        val_img_loader
    )


    val_acc = accuracy_score(

        y_val,

        p_val
    )


    print(

        f"Epoch {epoch}: "

        f"ValAcc={val_acc * 100:.2f}%"
    )


# ============================================================
# 9) FINAL TRAIN / VALIDATION / TEST EVALUATION
#    WITH BOOTSTRAP 95% CONFIDENCE INTERVAL
# ============================================================


print(

    "\n"

    + "=" * 90
)


print(

    "FINAL DATASET PERFORMANCE WITH 95% CONFIDENCE INTERVAL"
)


print(

    "=" * 90
)


# ------------------------------------------------------------
# TRAINING SET
# ------------------------------------------------------------

print(

    "\nEvaluating Training Set..."
)


y_train, p_train, prob_train = predict_on_loader(

    train_eval_loader
)


train_acc, train_lo, train_hi = bootstrap_accuracy_ci(

    y_train,

    p_train,

    n_boot=NUM_BOOT,

    alpha=ALPHA,

    seed=SEED
)


# ------------------------------------------------------------
# VALIDATION SET
# ------------------------------------------------------------

print(

    "\nEvaluating Validation Set..."
)


y_val, p_val, prob_val = predict_on_loader(

    val_img_loader
)


val_acc, val_lo, val_hi = bootstrap_accuracy_ci(

    y_val,

    p_val,

    n_boot=NUM_BOOT,

    alpha=ALPHA,

    seed=SEED
)


# ------------------------------------------------------------
# TEST SET
# ------------------------------------------------------------

print(

    "\nEvaluating Test Set..."
)


yt, pt, probt = predict_on_loader(

    test_img_loader
)


test_acc, test_lo, test_hi = bootstrap_accuracy_ci(

    yt,

    pt,

    n_boot=NUM_BOOT,

    alpha=ALPHA,

    seed=SEED
)


# ============================================================
# PRINT OVERALL ACCURACY TABLE
# ============================================================

print(

    "\n"

    + "=" * 80
)


print(

    f"{'Dataset':<20} | "
    f"{'Accuracy (%)':<20} | "
    f"{'95% Confidence Interval':<30}"
)


print(

    "-" * 80
)


print(

    f"{'Training':<20} | "

    f"{train_acc * 100:.2f} | "

    f"({train_lo * 100:.2f} - {train_hi * 100:.2f})"
)


print(

    f"{'Validation':<20} | "

    f"{val_acc * 100:.2f} | "

    f"({val_lo * 100:.2f} - {val_hi * 100:.2f})"
)


print(

    f"{'Testing':<20} | "

    f"{test_acc * 100:.2f} | "

    f"({test_lo * 100:.2f} - {test_hi * 100:.2f})"
)


print(

    "=" * 80
)


# ============================================================
# 10) TEST SET PER-CLASS 95% CI
# ============================================================

per_class_results = compute_per_class_metrics_with_ci(

    yt,

    pt,

    probt,

    class_names=full_dataset.classes,

    n_boot=NUM_BOOT,

    alpha=ALPHA,

    seed=SEED
)


# ============================================================
# PRINT PER-CLASS RESULTS
# ============================================================

print(

    "\n"

    + "=" * 115
)


print(

    f"{'Table: Evaluated each class performance with Precision, Recall, F-1 score, AUC, and Support (95% CI).':^115}"
)


print(

    "=" * 115
)


header = (

    f"{'Class':<15} | "

    f"{'Precision (95% CI)':<22} | "

    f"{'Recall (95% CI)':<22} | "

    f"{'F-1 score (95% CI)':<22} | "

    f"{'AUC (95% CI)':<22} | "

    f"{'Support':<8}"
)


print(header)


print(

    "-" * len(header)
)


for res in per_class_results:

    c = res["Class"]


    p_m, p_l, p_h = res[
        "Precision"
    ]


    r_m, r_l, r_h = res[
        "Recall"
    ]


    f_m, f_l, f_h = res[
        "F1"
    ]


    a_m, a_l, a_h = res[
        "AUC"
    ]


    supp = res[
        "Support"
    ]


    p_str = (

        f"{p_m:.2f} "

        f"({p_l:.2f}-{p_h:.2f})"
    )


    r_str = (

        f"{r_m:.2f} "

        f"({r_l:.2f}-{r_h:.2f})"
    )


    f_str = (

        f"{f_m:.2f} "

        f"({f_l:.2f}-{f_h:.2f})"
    )


    a_str = (

        f"{a_m:.2f} "

        f"({a_l:.2f}-{a_h:.2f})"
    )


    print(

        f"{c:<15} | "

        f"{p_str:<22} | "

        f"{r_str:<22} | "

        f"{f_str:<22} | "

        f"{a_str:<22} | "

        f"{supp:<8}"
    )


print(

    "=" * 115
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    yt,

    pt
)


print(

    "\nConfusion Matrix:\n"
)


print(cm)

Classes: ['Aair', 'Black Roi', 'Boal', 'Chapila', 'Common Carp', 'DeshiPuti', 'Foli', 'Grass Carp', 'Ilish', 'KalBaush', 'Katla', 'Koi', 'Long-whiskered Catfish', 'Magur', 'Mirror Carp', 'Mrigel', 'Nile Telapiya', 'Pabda', 'Pangas', 'Puti', 'Rui', 'Shol', 'Shorputi', 'Silver Carp', 'Taki', 'Tarabaim', 'Telapiya', 'Tengra']
Total images: 31608
Split: train=20228 val=5058 test=6322
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 214MB/s] 


ConvNeXt feature map: (768, 7, 7)


Epoch 1/10:  34%|███▎      | 213/633 [01:21<02:08,  3.26it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Epoch 1/10:  43%|████▎     | 274/633 [01:44<02:08,  2.79it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Epoch 1/10:  46%|████▌     | 289/633 [01:50<01:58,  2.89it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (92415825 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Epoch 1/10:  75%|███████▌  | 476/633 [03:06<00:52,  2.97it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96378672 pixels) exceeds limit of 89478485

Epoch 1: TrainLoss=0.6850


Predict:   1%|          | 5/633 [00:01<02:59,  3.50it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Predict:  18%|█▊        | 117/633 [00:42<02:09,  3.98it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (92415825 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Predict:  26%|██▌       | 166/633 [00:58<02:00,  3.88it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96378672 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Predict:  41%|████▏     | 262/633 [01:30<01:34,  3.93it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could

... (59 outputs left out)

Predict:  11%|█▏        | 18/159 [00:05<00:36,  3.88it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Predict:  34%|███▍      | 54/159 [00:18<00:32,  3.27it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Predict: 100%|██████████| 159/159 [00:52<00:00,  3.02it/s]



Evaluating Test Set...


Predict:   5%|▍         | 9/198 [00:03<01:05,  2.87it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Predict:  18%|█▊        | 35/198 [00:15<00:52,  3.08it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Predict:  59%|█████▉    | 117/198 [00:50<00:35,  2.31it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (91438802 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Predict: 100%|██████████| 198/198 [01:26<00:00,  2.29it/s]



Dataset              | Accuracy (%)         | 95% Confidence Interval       
--------------------------------------------------------------------------------
Training             | 99.95 | (99.91 - 99.98)
Validation           | 99.15 | (98.87 - 99.39)
Testing              | 99.05 | (98.81 - 99.29)

       Table: Evaluated each class performance with Precision, Recall, F-1 score, AUC, and Support (95% CI).       
Class           | Precision (95% CI)     | Recall (95% CI)        | F-1 score (95% CI)     | AUC (95% CI)           | Support 
------------------------------------------------------------------------------------------------------------------------------
Aair            | 1.00 (0.99-1.00)       | 1.00 (1.00-1.00)       | 1.00 (1.00-1.00)       | 1.00 (1.00-1.00)       | 361     
Black Roi       | 1.00 (1.00-1.00)       | 0.89 (0.80-0.96)       | 0.94 (0.89-0.98)       | 1.00 (0.99-1.00)       | 61      
Boal            | 0.99 (0.97-1.00)       | 0.99 (0.98-1.00)       | 0.99 (0

In [4]:
!pip install -q thop

In [5]:
# ============================================================
# FLOPs, PARAMETERS, MEMORY, INFERENCE TIME AND ACCURACY
# ============================================================

import time
import torch
import torch.nn as nn
import numpy as np

from thop import profile
from sklearn.metrics import accuracy_score


# ============================================================
# WRAPPER FOR CONVNEXT
# ============================================================

class ConvNeXtWrapper(nn.Module):

    def __init__(self, model):

        super().__init__()

        self.model = model

    def forward(self, x):

        return self.model(x)


# ============================================================
# CALCULATE CONVNEXT FLOPs
# ============================================================

def calculate_convnext_flops(
    convnext_model,
    device,
    input_size=(1, 3, 224, 224)
):

    convnext_model.eval()

    dummy_input = torch.randn(
        *input_size
    ).to(device)

    try:

        flops, params = profile(

            convnext_model,

            inputs=(dummy_input,),

            verbose=False
        )

        return flops, params

    except Exception as e:

        print(
            "ConvNeXt FLOPs calculation error:",
            e
        )

        return 0, 0


# ============================================================
# ESTIMATE GCN FLOPs
# ============================================================

def calculate_gcn_flops(
    num_nodes,
    num_edges,
    input_dim,
    hidden_dim,
    num_classes
):

    # --------------------------------------------------------
    # GCN LAYER 1
    # Approximation:
    #
    # Linear transformation:
    # Nodes × InputDim × HiddenDim
    #
    # Message passing:
    # Edges × HiddenDim
    # --------------------------------------------------------

    gcn1_linear = (

        num_nodes

        * input_dim

        * hidden_dim
    )


    gcn1_message = (

        num_edges

        * hidden_dim
    )


    # --------------------------------------------------------
    # GCN LAYER 2
    # --------------------------------------------------------

    gcn2_linear = (

        num_nodes

        * hidden_dim

        * hidden_dim
    )


    gcn2_message = (

        num_edges

        * hidden_dim
    )


    # --------------------------------------------------------
    # GLOBAL MEAN POOLING
    # --------------------------------------------------------

    pooling_flops = (

        num_nodes

        * hidden_dim
    )


    # --------------------------------------------------------
    # FINAL CLASSIFICATION LAYER
    # --------------------------------------------------------

    classifier_flops = (

        hidden_dim

        * num_classes
    )


    # --------------------------------------------------------
    # TOTAL
    # --------------------------------------------------------

    total_flops = (

        gcn1_linear

        + gcn1_message

        + gcn2_linear

        + gcn2_message

        + pooling_flops

        + classifier_flops
    )


    return total_flops


# ============================================================
# MAIN PERFORMANCE EVALUATION
# ============================================================

def evaluate_performance_efficiency(

    convnext_model,

    gcn_model,

    test_loader,

    device,

    model_name="ConvNeXt + GCN",

    num_warmup=10

):

    print("\n" + "=" * 110)

    print(
        "Performance & Efficiency Metrics Evaluation".center(110)
    )

    print("=" * 110)


    # ========================================================
    # EVALUATION MODE
    # ========================================================

    convnext_model.eval()

    gcn_model.eval()


    # ========================================================
    # CALCULATE PARAMETERS
    # ========================================================

    convnext_params = sum(

        p.numel()

        for p in convnext_model.parameters()
    )


    gcn_params = sum(

        p.numel()

        for p in gcn_model.parameters()
    )


    total_params = (

        convnext_params

        + gcn_params
    )


    params_million = (

        total_params

        / 1e6
    )


    print(
        f"\nConvNeXt Parameters: "
        f"{convnext_params / 1e6:.2f} M"
    )


    print(
        f"GCN Parameters: "
        f"{gcn_params / 1e6:.2f} M"
    )


    print(
        f"Total Parameters: "
        f"{params_million:.2f} M"
    )


    # ========================================================
    # CALCULATE CONVNEXT FLOPs
    # ========================================================

    print(
        "\nCalculating ConvNeXt FLOPs..."
    )


    convnext_flops, _ = calculate_convnext_flops(

        convnext_model,

        device,

        input_size=(1, 3, 224, 224)
    )


    print(
        f"ConvNeXt FLOPs: "
        f"{convnext_flops / 1e9:.4f} GFLOPs"
    )


    # ========================================================
    # CALCULATE GRAPH INFORMATION
    # ========================================================

    num_nodes = Hf * Wf

    num_edges = EDGE_INDEX_CPU.shape[1]


    print(
        f"\nGraph Nodes: {num_nodes}"
    )


    print(
        f"Graph Edges: {num_edges}"
    )


    # ========================================================
    # CALCULATE GCN FLOPs
    # ========================================================

    gcn_flops = calculate_gcn_flops(

        num_nodes=num_nodes,

        num_edges=num_edges,

        input_dim=C,

        hidden_dim=HIDDEN_DIM,

        num_classes=num_classes
    )


    print(
        f"GCN FLOPs: "
        f"{gcn_flops / 1e9:.6f} GFLOPs"
    )


    # ========================================================
    # TOTAL FLOPs
    # ========================================================

    total_flops = (

        convnext_flops

        + gcn_flops
    )


    total_gflops = (

        total_flops

        / 1e9
    )


    print(
        f"Total FLOPs: "
        f"{total_gflops:.4f} GFLOPs"
    )


    # ========================================================
    # RESET GPU MEMORY
    # ========================================================

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats(
            device
        )


    # ========================================================
    # WARM-UP
    # ========================================================

    print(
        "\nPerforming GPU warm-up..."
    )


    warmup_count = 0


    with torch.no_grad():

        for imgs, ys in test_loader:

            imgs = imgs.to(

                device,

                non_blocking=True
            )


            ys = ys.to(

                device,

                non_blocking=True
            )


            feats = convnext_model(
                imgs
            )


            graphs = features_to_graph_batch(

                feats.cpu(),

                ys.cpu()
            )


            batch = Batch.from_data_list(

                graphs

            ).to(device)


            _ = gcn_model(
                batch
            )


            warmup_count += 1


            if warmup_count >= num_warmup:

                break


    # ========================================================
    # ACCURACY AND INFERENCE TIME
    # ========================================================

    all_true = []

    all_pred = []

    total_inference_time = 0.0

    total_samples = 0


    print(
        "\nEvaluating Accuracy and Inference Time..."
    )


    with torch.no_grad():

        for imgs, ys in tqdm(

            test_loader,

            desc="Performance Evaluation"
        ):

            imgs = imgs.to(

                device,

                non_blocking=True
            )


            ys = ys.to(

                device,

                non_blocking=True
            )


            # Synchronize GPU before timing

            if torch.cuda.is_available():

                torch.cuda.synchronize()


            start_time = time.perf_counter()


            # =================================================
            # CONVNEXT FORWARD PASS
            # =================================================

            feats = convnext_model(
                imgs
            )


            # =================================================
            # FEATURE MAP TO GRAPH
            # =================================================

            graphs = features_to_graph_batch(

                feats.cpu(),

                ys.cpu()
            )


            batch = Batch.from_data_list(

                graphs

            ).to(device)


            # =================================================
            # GCN FORWARD PASS
            # =================================================

            logits = gcn_model(
                batch
            )


            preds = torch.argmax(

                logits,

                dim=1
            )


            # Synchronize GPU after inference

            if torch.cuda.is_available():

                torch.cuda.synchronize()


            end_time = time.perf_counter()


            # =================================================
            # CALCULATE TIME
            # =================================================

            batch_time = (

                end_time

                - start_time
            )


            total_inference_time += (

                batch_time
            )


            total_samples += (

                imgs.size(0)
            )


            # =================================================
            # SAVE PREDICTIONS
            # =================================================

            all_true.extend(

                ys.cpu().numpy()
            )


            all_pred.extend(

                preds.cpu().numpy()
            )


    # ========================================================
    # ACCURACY
    # ========================================================

    accuracy = accuracy_score(

        all_true,

        all_pred

    ) * 100


    # ========================================================
    # INFERENCE TIME PER IMAGE
    # ========================================================

    inference_time_ms = (

        total_inference_time

        / total_samples

    ) * 1000


    # ========================================================
    # GPU MEMORY
    # ========================================================

    if torch.cuda.is_available():

        memory_usage_mb = (

            torch.cuda.max_memory_allocated(

                device

            )

            / (1024 ** 2)
        )

    else:

        memory_usage_mb = 0.0


    # ========================================================
    # FINAL TABLE
    # ========================================================

    print("\n" + "=" * 120)

    print(
        "Performance & Efficiency Metrics Evaluation".center(120)
    )

    print("=" * 120)


    header = (

        f"{'Method':<20} | "

        f"{'FLOPs (GFLOPs)':<18} | "

        f"{'Parameters (M)':<18} | "

        f"{'Memory Usage (MB)':<20} | "

        f"{'Inference Time (ms)':<22} | "

        f"{'Accuracy (%)':<15}"
    )


    print(header)

    print("-" * len(header))


    print(

        f"{model_name:<20} | "

        f"{total_gflops:<18.4f} | "

        f"{params_million:<18.2f} | "

        f"{memory_usage_mb:<20.2f} | "

        f"{inference_time_ms:<22.2f} | "

        f"{accuracy:<15.2f}"
    )


    print("=" * 120)


    # ========================================================
    # RETURN RESULTS
    # ========================================================

    return {

        "Method": model_name,

        "FLOPs (GFLOPs)": total_gflops,

        "ConvNeXt FLOPs": (

            convnext_flops / 1e9
        ),

        "GCN FLOPs": (

            gcn_flops / 1e9
        ),

        "Parameters (M)": params_million,

        "Memory Usage (MB)": memory_usage_mb,

        "Inference Time (ms)": inference_time_ms,

        "Accuracy (%)": accuracy
    }

In [6]:
# ============================================================
# FINAL PERFORMANCE EVALUATION
# ============================================================

performance_results = evaluate_performance_efficiency(

    convnext_model=convnext_model,

    gcn_model=gcn_model,

    test_loader=test_img_loader,

    device=device,

    model_name="ConvNeXt + GCN",

    num_warmup=10
)


                                 Performance & Efficiency Metrics Evaluation                                  

ConvNeXt Parameters: 27.82 M
GCN Parameters: 0.12 M
Total Parameters: 27.94 M

Calculating ConvNeXt FLOPs...
ConvNeXt FLOPs: 4.4633 GFLOPs

Graph Nodes: 49
Graph Edges: 168
GCN FLOPs: 0.005673 GFLOPs
Total FLOPs: 4.4690 GFLOPs

Performing GPU warm-up...


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(



Evaluating Accuracy and Inference Time...


Performance Evaluation:   5%|▍         | 9/198 [00:02<00:43,  4.33it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Performance Evaluation:  18%|█▊        | 35/198 [00:12<00:40,  3.99it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Performance Evaluation:  59%|█████▉    | 117/198 [00:40<00:24,  3.25it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (91438802 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Performance Evaluation: 100%|██████████| 198/198 [01:06<00:00,  2.97it/s]


                                      Performance & Efficiency Metrics Evaluation                                       
Method               | FLOPs (GFLOPs)     | Parameters (M)     | Memory Usage (MB)    | Inference Time (ms)    | Accuracy (%)   
--------------------------------------------------------------------------------------------------------------------------------
ConvNeXt + GCN       | 4.4690             | 27.94              | 529.37               | 4.05                   | 99.05          


In [10]:
# ============================================================
# SAVE TRAINED HYBRID MODEL
# FORMATS: .PT AND .PKL
# ============================================================

import os
import torch
import pickle


# ------------------------------------------------------------
# CREATE SAVE DIRECTORY
# ------------------------------------------------------------

SAVE_DIR = "/kaggle/working/saved_models"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


# ============================================================
# PREPARE MODEL CHECKPOINT
# ============================================================

checkpoint = {

    # --------------------------------------------------------
    # MODEL WEIGHTS
    # --------------------------------------------------------

    "convnext_state_dict":
        convnext_model.state_dict(),

    "gcn_state_dict":
        gcn_model.state_dict(),


    # --------------------------------------------------------
    # MODEL CONFIGURATION
    # --------------------------------------------------------

    "input_dim":
        C,

    "hidden_dim":
        HIDDEN_DIM,

    "num_classes":
        num_classes,

    "dropout":
        DROPOUT,


    # --------------------------------------------------------
    # DATASET INFORMATION
    # --------------------------------------------------------

    "class_names":
        full_dataset.classes,


    # --------------------------------------------------------
    # IMAGE INFORMATION
    # --------------------------------------------------------

    "image_size":
        224,

    "feature_height":
        Hf,

    "feature_width":
        Wf
}


# ============================================================
# SAVE AS .PT
# ============================================================

pt_path = os.path.join(
    SAVE_DIR,
    "ConvNeXt_GCN_Fish_Classifier.pt"
)

torch.save(
    checkpoint,
    pt_path
)

print(f"✓ PT model saved: {pt_path}")


# ============================================================
# SAVE AS .PKL
# ============================================================

pkl_path = os.path.join(
    SAVE_DIR,
    "ConvNeXt_GCN_Fish_Classifier.pkl"
)

with open(
    pkl_path,
    "wb"
) as f:

    pickle.dump(
        checkpoint,
        f
    )


print(f"✓ PKL model saved: {pkl_path}")


print("\n" + "=" * 70)
print("MODEL SAVING COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nSaved files:")
print(f"1. {pt_path}")
print(f"2. {pkl_path}")

✓ PT model saved: /kaggle/working/saved_models/ConvNeXt_GCN_Fish_Classifier.pt
✓ PKL model saved: /kaggle/working/saved_models/ConvNeXt_GCN_Fish_Classifier.pkl

MODEL SAVING COMPLETED SUCCESSFULLY

Saved files:
1. /kaggle/working/saved_models/ConvNeXt_GCN_Fish_Classifier.pt
2. /kaggle/working/saved_models/ConvNeXt_GCN_Fish_Classifier.pkl


In [12]:
# ============================================================
# SAVE MASTER HYBRID MODEL
# FP32 - HIGH ACCURACY VERSION
# ============================================================

import os
import torch
import pickle
import json

SAVE_DIR = "/kaggle/working/saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# MAKE SURE MODELS ARE IN EVALUATION MODE
# ------------------------------------------------------------

convnext_model.eval()
gcn_model.eval()

# ------------------------------------------------------------
# ENSURE FP32 WEIGHTS
# ------------------------------------------------------------

convnext_model = convnext_model.float()
gcn_model = gcn_model.float()

# ------------------------------------------------------------
# CHECKPOINT
# ------------------------------------------------------------

checkpoint = {

    # Model weights
    "convnext_state_dict":
        convnext_model.state_dict(),

    "gcn_state_dict":
        gcn_model.state_dict(),

    # Architecture configuration
    "input_dim":
        C,

    "hidden_dim":
        HIDDEN_DIM,

    "num_classes":
        num_classes,

    "dropout":
        DROPOUT,

    # Dataset
    "class_names":
        list(full_dataset.classes),

    # Image configuration
    "image_size":
        224,

    "feature_height":
        Hf,

    "feature_width":
        Wf,

    # Deployment information
    "dtype":
        "float32",

    "model_name":
        "ConvNeXt_GCN_Fish_Classifier",

    "version":
        "1.0",

    "framework":
        "PyTorch",

    "deployment_targets":
        [
            "android",
            "web"
        ]
}

# ============================================================
# SAVE MASTER .PT
# ============================================================

pt_path = os.path.join(
    SAVE_DIR,
    "ConvNeXt_GCN_Fish_Classifier_MASTER_FP32.pt"
)

torch.save(
    checkpoint,
    pt_path
)

print("✓ Master PT saved:")
print(pt_path)


# ============================================================
# SAVE PKL BACKUP
# ============================================================

pkl_path = os.path.join(
    SAVE_DIR,
    "ConvNeXt_GCN_Fish_Classifier_MASTER_FP32.pkl"
)

with open(pkl_path, "wb") as f:
    pickle.dump(checkpoint, f)

print("✓ PKL backup saved:")
print(pkl_path)


# ============================================================
# SAVE CONFIGURATION AS JSON
# ============================================================

config = {
    "model_name": "ConvNeXt_GCN_Fish_Classifier",
    "version": "1.0",
    "input_dim": int(C),
    "hidden_dim": int(HIDDEN_DIM),
    "num_classes": int(num_classes),
    "dropout": float(DROPOUT),
    "class_names": list(full_dataset.classes),
    "image_size": 224,
    "feature_height": int(Hf),
    "feature_width": int(Wf),
    "dtype": "float32"
}

json_path = os.path.join(
    SAVE_DIR,
    "ConvNeXt_GCN_Fish_Classifier_config.json"
)

with open(json_path, "w") as f:
    json.dump(config, f, indent=4)

print("✓ JSON configuration saved:")
print(json_path)


print("\n" + "=" * 70)
print("MASTER MODEL SAVING COMPLETED")
print("=" * 70)

✓ Master PT saved:
/kaggle/working/saved_models/ConvNeXt_GCN_Fish_Classifier_MASTER_FP32.pt
✓ PKL backup saved:
/kaggle/working/saved_models/ConvNeXt_GCN_Fish_Classifier_MASTER_FP32.pkl
✓ JSON configuration saved:
/kaggle/working/saved_models/ConvNeXt_GCN_Fish_Classifier_config.json

MASTER MODEL SAVING COMPLETED


In [13]:
# ============================================================
# LOAD MASTER MODEL
# ============================================================

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = torch.load(
    "/kaggle/working/saved_models/"
    "ConvNeXt_GCN_Fish_Classifier_MASTER_FP32.pt",
    map_location=device,
    weights_only=False
)

print("Checkpoint loaded.")

print("Classes:")
print(checkpoint["class_names"])

print("Number of classes:",
      checkpoint["num_classes"])

print("Data type:",
      checkpoint["dtype"])

Checkpoint loaded.
Classes:
['Aair', 'Black Roi', 'Boal', 'Chapila', 'Common Carp', 'DeshiPuti', 'Foli', 'Grass Carp', 'Ilish', 'KalBaush', 'Katla', 'Koi', 'Long-whiskered Catfish', 'Magur', 'Mirror Carp', 'Mrigel', 'Nile Telapiya', 'Pabda', 'Pangas', 'Puti', 'Rui', 'Shol', 'Shorputi', 'Silver Carp', 'Taki', 'Tarabaim', 'Telapiya', 'Tengra']
Number of classes: 28
Data type: float32


In [14]:
convnext_model.load_state_dict(
    checkpoint["convnext_state_dict"]
)

gcn_model.load_state_dict(
    checkpoint["gcn_state_dict"]
)

convnext_model = convnext_model.float().to(device)
gcn_model = gcn_model.float().to(device)

convnext_model.eval()
gcn_model.eval()

GCNClassifier(
  (conv1): GCNConv(768, 128)
  (conv2): GCNConv(128, 128)
  (lin): Linear(in_features=128, out_features=28, bias=True)
)

In [22]:

# ============================================================
# COMPLETE SINGLE IMAGE INFERENCE
# ConvNeXt Feature Map -> Graph -> GCN
# ============================================================

import os
import torch
import torch.nn.functional as F

from PIL import Image
from torchvision import transforms

# PyTorch Geometric
from torch_geometric.data import Data


# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("DEVICE")
print("=" * 70)

print("Device:", device)


# ============================================================
# 2. IMAGE PATH
# ============================================================

IMAGE_PATH = (
    "/kaggle/input/datasets/mdhasanahmad/fresh-water-dataset/Original Image/Rui/Rohu (1).jpg"
    
)


# ============================================================
# 3. CHECK IMAGE
# ============================================================

if not os.path.isfile(IMAGE_PATH):

    raise FileNotFoundError(
        f"\nImage not found:\n{IMAGE_PATH}"
    )

print("\nImage found:")
print(IMAGE_PATH)


# ============================================================
# 4. LOAD IMAGE
# ============================================================

image = Image.open(
    IMAGE_PATH
).convert("RGB")

print(
    "Original image size:",
    image.size
)


# ============================================================
# 5. VALIDATION TRANSFORM
# ============================================================
#
# IMPORTANT:
# Use EXACTLY the same validation preprocessing used during
# model training whenever possible.
#
# This is the standard ImageNet preprocessing for ConvNeXt.
#

val_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# 6. IMAGE -> TENSOR
# ============================================================

image_tensor = val_transform(
    image
)

# Add batch dimension
#
# [3,224,224]
#       ↓
# [1,3,224,224]

image_tensor = image_tensor.unsqueeze(0)

image_tensor = image_tensor.to(device)

print(
    "\nInput tensor shape:",
    image_tensor.shape
)


# ============================================================
# 7. CHECK MODELS
# ============================================================

if "convnext_model" not in globals():

    raise NameError(
        "\nconvnext_model is not defined.\n"
        "Run the cell where your trained ConvNeXt model "
        "was loaded."
    )


if "gcn_model" not in globals():

    raise NameError(
        "\ngcn_model is not defined.\n"
        "Run the cell where your trained GCN model "
        "was loaded."
    )


# ============================================================
# 8. MOVE MODELS TO DEVICE
# ============================================================

convnext_model = convnext_model.to(
    device
)

gcn_model = gcn_model.to(
    device
)


# ============================================================
# 9. EVALUATION MODE
# ============================================================

convnext_model.eval()

gcn_model.eval()


# ============================================================
# 10. FUNCTION:
#     FEATURE MAP -> GRAPH
# ============================================================

def feature_map_to_graph(
    feature_map,
    k=8
):

    """
    Convert ConvNeXt feature map into a
    PyTorch Geometric graph.

    Input:
        feature_map:
            [1, C, H, W]

    Output:
        PyG Data object

    Nodes:
        H * W

    Node features:
        C-dimensional ConvNeXt features

    Edges:
        KNN cosine-similarity edges
        +
        local spatial edges
    """

    # --------------------------------------------------------
    # Remove batch dimension
    # --------------------------------------------------------

    x = feature_map.squeeze(0)

    # x:
    # [C,H,W]

    C, H, W = x.shape


    # --------------------------------------------------------
    # Convert spatial positions into nodes
    # --------------------------------------------------------
    #
    # [C,H,W]
    #       ↓
    # [H*W,C]
    #

    x = x.permute(
        1,
        2,
        0
    ).contiguous()

    x = x.view(
        H * W,
        C
    )


    # --------------------------------------------------------
    # L2 NORMALIZATION
    # --------------------------------------------------------

    x_norm = F.normalize(
        x,
        p=2,
        dim=1
    )


    # --------------------------------------------------------
    # COSINE SIMILARITY
    # --------------------------------------------------------

    similarity = torch.mm(
        x_norm,
        x_norm.t()
    )


    # --------------------------------------------------------
    # REMOVE SELF CONNECTION
    # --------------------------------------------------------

    similarity.fill_diagonal_(
        -float("inf")
    )


    # --------------------------------------------------------
    # KNN EDGES
    # --------------------------------------------------------

    k = min(
        k,
        H * W - 1
    )

    _, knn_indices = torch.topk(
        similarity,
        k=k,
        dim=1
    )


    # --------------------------------------------------------
    # CREATE KNN EDGE LIST
    # --------------------------------------------------------

    source_nodes = (
        torch.arange(
            H * W,
            device=x.device
        )
        .unsqueeze(1)
        .expand(-1, k)
        .reshape(-1)
    )

    target_nodes = (
        knn_indices
        .reshape(-1)
    )


    # --------------------------------------------------------
    # LOCAL SPATIAL EDGES
    # --------------------------------------------------------

    spatial_edges = []


    for row in range(H):

        for col in range(W):

            node = (
                row * W
                + col
            )


            # -----------------------------------------------
            # Up
            # -----------------------------------------------

            if row > 0:

                neighbor = (
                    (row - 1) * W
                    + col
                )

                spatial_edges.append(
                    [node, neighbor]
                )


            # -----------------------------------------------
            # Down
            # -----------------------------------------------

            if row < H - 1:

                neighbor = (
                    (row + 1) * W
                    + col
                )

                spatial_edges.append(
                    [node, neighbor]
                )


            # -----------------------------------------------
            # Left
            # -----------------------------------------------

            if col > 0:

                neighbor = (
                    row * W
                    + (col - 1)
                )

                spatial_edges.append(
                    [node, neighbor]
                )


            # -----------------------------------------------
            # Right
            # -----------------------------------------------

            if col < W - 1:

                neighbor = (
                    row * W
                    + (col + 1)
                )

                spatial_edges.append(
                    [node, neighbor]
                )


    # --------------------------------------------------------
    # CONVERT SPATIAL EDGES
    # --------------------------------------------------------

    if len(spatial_edges) > 0:

        spatial_edges = torch.tensor(
            spatial_edges,
            dtype=torch.long,
            device=x.device
        ).t().contiguous()

    else:

        spatial_edges = torch.empty(
            (2, 0),
            dtype=torch.long,
            device=x.device
        )


    # --------------------------------------------------------
    # KNN EDGE INDEX
    # --------------------------------------------------------

    knn_edge_index = torch.stack(
        [
            source_nodes,
            target_nodes
        ],
        dim=0
    )


    # --------------------------------------------------------
    # COMBINE KNN + SPATIAL EDGES
    # --------------------------------------------------------

    edge_index = torch.cat(
        [
            knn_edge_index,
            spatial_edges
        ],
        dim=1
    )


    # --------------------------------------------------------
    # ADD REVERSE EDGES
    # --------------------------------------------------------

    reverse_edges = edge_index[
        [1, 0],
        :
    ]

    edge_index = torch.cat(
        [
            edge_index,
            reverse_edges
        ],
        dim=1
    )


    # --------------------------------------------------------
    # REMOVE DUPLICATE EDGES
    # --------------------------------------------------------

    edge_index = torch.unique(
        edge_index,
        dim=1
    )


    # --------------------------------------------------------
    # CREATE PYTORCH GEOMETRIC DATA
    # --------------------------------------------------------

    data = Data(
        x=x,
        edge_index=edge_index
    )


    return data


# ============================================================
# 11. INFERENCE
# ============================================================

with torch.no_grad():

    # --------------------------------------------------------
    # STEP 1:
    # ConvNeXt feature extraction
    # --------------------------------------------------------

    conv_features = convnext_model(
        image_tensor
    )


    print(
        "\nConvNeXt output shape:",
        conv_features.shape
    )


    # --------------------------------------------------------
    # STEP 2:
    # Convert feature map to graph
    # --------------------------------------------------------

    graph_data = feature_map_to_graph(
        conv_features,
        k=8
    )


    # Move graph to device

    graph_data = graph_data.to(
        device
    )


    # --------------------------------------------------------
    # GRAPH INFORMATION
    # --------------------------------------------------------

    print(
        "Graph node feature shape:",
        graph_data.x.shape
    )

    print(
        "Graph edge index shape:",
        graph_data.edge_index.shape
    )

    print(
        "Number of nodes:",
        graph_data.num_nodes
    )

    print(
        "Number of edges:",
        graph_data.num_edges
    )


    # --------------------------------------------------------
    # STEP 3:
    # GCN
    # --------------------------------------------------------

    output = gcn_model(
        graph_data
    )


    # --------------------------------------------------------
    # STEP 4:
    # HANDLE DIFFERENT OUTPUT TYPES
    # --------------------------------------------------------

    if isinstance(
        output,
        tuple
    ):

        output = output[0]


    elif isinstance(
        output,
        dict
    ):

        # Try common names

        if "logits" in output:

            output = output["logits"]

        elif "out" in output:

            output = output["out"]

        else:

            raise RuntimeError(
                "GCN returned a dictionary but "
                "no 'logits' or 'out' key was found."
            )


    # --------------------------------------------------------
    # PRINT OUTPUT
    # --------------------------------------------------------

    print(
        "Final output shape:",
        output.shape
    )


    # --------------------------------------------------------
    # STEP 5:
    # SOFTMAX
    # --------------------------------------------------------

    probabilities = F.softmax(
        output,
        dim=1
    )


    # --------------------------------------------------------
    # STEP 6:
    # TOP PREDICTION
    # --------------------------------------------------------

    confidence, predicted_class = torch.max(
        probabilities,
        dim=1
    )


# ============================================================
# 12. LOAD CLASS NAMES
# ============================================================

class_names = None


# ------------------------------------------------------------
# Try checkpoint
# ------------------------------------------------------------

if "checkpoint" in globals():

    if isinstance(
        checkpoint,
        dict
    ):

        if "class_names" in checkpoint:

            class_names = checkpoint[
                "class_names"
            ]


# ------------------------------------------------------------
# Try global class_names
# ------------------------------------------------------------

if class_names is None:

    if "class_names" in globals():

        class_names = globals()[
            "class_names"
        ]


# ------------------------------------------------------------
# Final check
# ------------------------------------------------------------

if class_names is None:

    raise NameError(
        "\nclass_names was not found.\n\n"
        "Define it manually, for example:\n\n"
        "class_names = ['Aair', 'Bair', 'Cair']\n\n"
        "using the EXACT class order used during training."
    )


# ============================================================
# 13. PREDICTION
# ============================================================

predicted_index = (
    predicted_class.item()
)

predicted_name = class_names[
    predicted_index
]

confidence_percentage = (
    confidence.item()
    * 100
)


# ============================================================
# 14. FINAL RESULT
# ============================================================

print("\n")
print("=" * 70)
print("                    PREDICTION RESULT")
print("=" * 70)

print(
    f"Image           : "
    f"{os.path.basename(IMAGE_PATH)}"
)

print(
    f"Predicted Class : "
    f"{predicted_name}"
)

print(
    f"Class Index     : "
    f"{predicted_index}"
)

print(
    f"Confidence      : "
    f"{confidence_percentage:.2f}%"
)

print("=" * 70)


# ============================================================
# 15. ALL CLASS PROBABILITIES
# ============================================================

print("\nALL CLASS PROBABILITIES")
print("-" * 70)

for i, probability in enumerate(
    probabilities[0]
):

    print(
        f"{class_names[i]:30s}"
        f" : "
        f"{probability.item() * 100:8.2f}%"
    )

print("-" * 70)


# ============================================================
# 16. TOP-3 PREDICTIONS
# ============================================================

top_k = min(
    3,
    probabilities.shape[1]
)

top_probabilities, top_indices = (
    torch.topk(
        probabilities,
        k=top_k,
        dim=1
    )
)


print("\nTOP-3 PREDICTIONS")
print("-" * 70)

for rank in range(top_k):

    idx = top_indices[
        0,
        rank
    ].item()

    prob = top_probabilities[
        0,
        rank
    ].item() * 100

    print(
        f"{rank + 1}. "
        f"{class_names[idx]:30s}"
        f" {prob:.2f}%"
    )

print("-" * 70)



DEVICE
Device: cuda

Image found:
/kaggle/input/datasets/mdhasanahmad/fresh-water-dataset/Original Image/Rui/Rohu (1).jpg
Original image size: (2033, 2710)

Input tensor shape: torch.Size([1, 3, 224, 224])

ConvNeXt output shape: torch.Size([1, 768, 7, 7])
Graph node feature shape: torch.Size([49, 768])
Graph edge index shape: torch.Size([2, 620])
Number of nodes: 49
Number of edges: 620
Final output shape: torch.Size([1, 28])


                    PREDICTION RESULT
Image           : Rohu (1).jpg
Predicted Class : Rui
Class Index     : 20
Confidence      : 95.23%

ALL CLASS PROBABILITIES
----------------------------------------------------------------------
Aair                           :     0.00%
Black Roi                      :     0.14%
Boal                           :     0.00%
Chapila                        :     0.00%
Common Carp                    :     0.00%
DeshiPuti                      :     0.00%
Foli                           :     0.00%
Grass Carp                     : 